# RAG PDF Chatbot Experiement Notebook
---

**Register your UV venv environment with Jupyter**
```
uv run python -m ipykernel install --user --name rag-env --display-name "Python (RAG)"
```

---

In [1]:
from dotenv import load_dotenv
load_dotenv("../.env")

import os
print("Gemini API Key Loaded: ", bool(os.getenv("GOOGLE_API_KEY")))


Gemini API Key Loaded:  True


## Configuration
---

In [2]:
os.getcwd()


'd:\\ROOT\\Projects\\RAG-PDF-Chatbot\\notebooks'

In [3]:
DOCS_BASE = "../data"
DOC_PATHS = [
  "attention_is_all_you_need_paper.pdf"
]


## Load Document
---

In [4]:
from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader


def load_documents(doc_paths: list[str] | None = None) -> list[Document]:
  paths = doc_paths or DOC_PATHS
  docs: list[Document] = []

  for path in paths:
    source = f"{DOCS_BASE}/{path}"
    pdf_loader = PyPDFLoader(source)
    pdf_extracted = pdf_loader.load() # extract text from pdf
    # pdf_extracted is list of documents i.e one page is one Document
    docs.extend(pdf_extracted)

  return docs


C:\Users\Vivek\AppData\Local\Temp\ipykernel_17804\727616756.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [5]:
docs = load_documents()
print(f"Loaded {len(docs)} documentation pages.")


Loaded 11 documentation pages.


In [6]:
docs[0]


Document(metadata={'producer': 'PyPDF2', 'creator': 'PyPDF', 'creationdate': '', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'publisher': 'Curran Associates, Inc.', 'language': 'en-US', 'created': '2017', 'eventtype': 'Poster', 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions entirely.Experiments on two machine translation tasks show these models to be superiorin quality while being more parallelizable and requiring significantly less timeto train. Our single model with 165 million parameters, achieves 27.5 BLEU onEnglish-to-German translation, improving over the existing best ensemble result by over 1 BLEU. On E

In [7]:
print("Page 7 first 500 characters: \n")
print(docs[6].page_content[:500])


Page 7 first 500 characters: 

the input sequence centered around the respective output position. This would increase the maximum
path length toO(n/r). We plan to investigate this approach further in future work.
A single convolutional layer with kernel widthk<n does not connect all pairs of input and output
positions. Doing so requires a stack ofO(n/k) convolutional layers in the case of contiguous kernels,
orO(logk(n)) in the case of dilated convolutions [ 15], increasing the length of the longest paths
between any two posi


In [8]:
total_chars = sum(len(doc.page_content) for doc in docs)
print(f"Total characters: {total_chars}")


Total characters: 32601


**Rule of thumb: 1 token ≈ 4 English characters**
- So 32 thousand characters is around 8000 tokens

## Splitting or Chunking the Document
---

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


def chunk_documents(raw_docs: list[Document]):
  text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=100)
  chunks = text_splitter.split_documents(raw_docs)
  return chunks


In [10]:
chunks = chunk_documents(docs)


In [11]:
chunks[0]


Document(metadata={'producer': 'PyPDF2', 'creator': 'PyPDF', 'creationdate': '', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'publisher': 'Curran Associates, Inc.', 'language': 'en-US', 'created': '2017', 'eventtype': 'Poster', 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions entirely.Experiments on two machine translation tasks show these models to be superiorin quality while being more parallelizable and requiring significantly less timeto train. Our single model with 165 million parameters, achieves 27.5 BLEU onEnglish-to-German translation, improving over the existing best ensemble result by over 1 BLEU. On E

In [12]:
print(f"Split documentation into {len(chunks)} chunks.")


Split documentation into 168 chunks.


In [13]:
print("Chunk 8 first 100 characters: \n")
print(chunks[7].page_content[:100])


Chunk 8 first 100 characters: 

ensembles, by over 2 BLEU. On the WMT 2014 English-to-French translation task,
our model establishes


In [14]:
total_chars = sum(len(chunk.page_content) for chunk in chunks)
print(f"Total characters: {total_chars}")


Total characters: 42902


**Notice the total characters increased after chunking**
- Becuase of chunk overlap

## Embedding Model
---

In [15]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings


embedding_model = GoogleGenerativeAIEmbeddings(model='models/gemini-embedding-001') # 100 RPM, 100 RPD, 30K TPM


In [16]:
embedding_model


GoogleGenerativeAIEmbeddings(client=<google.genai.client.Client object at 0x00000209C7FA6CF0>, model='models/gemini-embedding-001', task_type=None, google_api_key=SecretStr('**********'), credentials=None, vertexai=None, project=None, location=None, base_url=None, additional_headers=None, client_args=None, api_version=None, request_options=None, output_dimensionality=None)

## Vector Store(run only once)
---

In [17]:
from langchain_chroma import Chroma


vector_store = Chroma(
  collection_name="first_collection",
  embedding_function=embedding_model,
  persist_directory="../chroma_db"
)


In [18]:
# Embedding in batches
# Cuz i'm using free tier: 100 RPM i.e 100 chunks per minute limit
# I have a total of 168 chunks

import time

batch_size = 100

for i in range(0, len(chunks), batch_size):
    batch = chunks[i:i + batch_size]

    vector_store.add_documents(batch)

    print(f"Added {len(batch)} chunks")

    if i + batch_size < len(chunks):
        print("Sleeping 60 secs...")
        time.sleep(60)

print("All Chunks indexed")


Added 100 chunks
Sleeping 60 secs...
Added 68 chunks
All Chunks indexed


**NOTE: do not rerun above as it'll repeat and duplicate the document embeddings**

## Retrieval and Tool
---

In [ ]:
num_results = 4
retriever = vector_store.as_retriever(search_kwargs={'k': num_results})


In [ ]:
retriever


VectorStoreRetriever(tags=['Chroma', 'GoogleGenerativeAIEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x00000209CAC256A0>, search_kwargs={'k': 4})

In [21]:
from langchain.tools import tool


@tool
def search_documents(query: str) -> str:
    """
    Search the uploaded documents for relevant information.
    Use this tool whenever the user asks questions
    related to the uploaded documents.
    """
    docs = retriever.invoke(query)

    if not docs:
        return "No relevant information found."

    results = []

    for doc in docs:
        results.append(doc.page_content)

    return "\n\n".join(results)


## LLM
---

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI


gemini_llm = ChatGoogleGenerativeAI(
  model='gemini-3.1-flash-lite',
  temperature=0
)

gemini_llm


ChatGoogleGenerativeAI(metadata={'lc_versions': {'langchain-core': '1.7.5', 'langchain': '1.4.8', 'langchain-google-genai': '4.4.2'}}, profile={'name': 'Gemini 3.1 Flash Lite', 'release_date': '2026-05-07', 'last_updated': '2026-05-07', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True, 'reasoning_effort_levels': ['minimal', 'low', 'medium', 'high'], 'reasoning_effort_default': 'minimal'}, google_api_key=SecretStr('**********'), model='gemini-3.1-flash-lite', temperature=0.0, client=<google.genai.client.Client object at 0x00000209CB5B1A90>, default_metadata=(), model_kwarg

## Agent
---

In [ ]:
from langchain.agents import create_agent


agent = create_agent(
  model=gemini_llm,
  tools=[search_documents],
  system_prompt="""
  You are a helpful document assistant.

  Instructions:
  1. Answer questions using the uploaded documents.
  2. Use the search_documents tool when document
      information is needed.
  3. Do not make up information.
  4. If the answer is not available in the documents,
      clearly say that you couldn't find it.
  5. Give clear and concise answers.
  """
) 


## Agent Chat Loop
---

In [28]:
while True:
    question = input("\nYou: ")

    if question.lower() == "exit":
        break

    response = agent.invoke({
        "messages": [
            {"role": "user", "content": question}
        ]
    })

    print("\nYou       :", question)
    print("\nAssistant :", response["messages"][-1].content[0]['text'])
    print("\n\n")



You       : Hello

Assistant : Hello! How can I help you today?




You       : What document are you loaded with?

Assistant : I am loaded with a document that contains a list of references to various research papers in the field of machine learning and neural networks, specifically focusing on topics like machine translation, attention mechanisms, and deep learning architectures (such as LSTMs and Xception).




You       : Give one sentence summary about the document

Assistant : This document introduces the Transformer, a novel sequence transduction model that replaces traditional recurrent layers with multi-headed self-attention to achieve superior performance in language tasks.




You       : Does attention mean human attention?

Assistant : Based on the provided documents, "attention" refers to a mathematical mechanism used in sequence modeling and transduction models, not human attention.

The documents define an attention function as a process that maps a query and a set of 

In [29]:
response["messages"][-1].content


[{'type': 'text',
  'text': 'I could not find information about the father of the Moon in the provided documents.',
  'extras': {'signature': 'EnMKcQFpFH0TJ+n9ZiO0jGxG5ZwR3i6Sl1/CfnUA1D5xD5Ueblp80shtIwlW8QtavaWSi1OM+kJi8YqSnsVYBM3g538QLf8OucyGthDO+odg60SvxrtC/BEaGFWMTPHDdsmbN5qVY6N7otMwyCTv0c5WVtu+'}}]